In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import pickle
from collections import Counter
from sklearn.metrics import precision_recall_curve, roc_auc_score, confusion_matrix, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from scipy.stats.mstats import gmean
import math
import datetime as dt
import matplotlib
font = {'size': 24}
matplotlib.rc('font', **font)
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
df_new = pd.read_stata("../degree_completion_1/df_new.dta")
predictors = pickle.load(open("../degree_completion_1/predictors.p", "rb"))

In [ ]:
program = pd.read_stata("student_deglvl.dta")
df_new = program.merge(df_new, on=['vccsid'], how='right')

train_df_all = df_new[df_new.valid == 0]
test_df_all = df_new[df_new.valid == 1]

In [ ]:
def calc_cw(y):
    # Calculate the weight of each letter grade to be used in the modeling fitting procedure: the weight is inversely proportional to the square root of the frequency of the letter grade in the training sample
    cw = Counter(y)
    class_weight = {k:np.sqrt(cw.most_common()[0][-1]/v, dtype=np.float32) for k,v in cw.items()}
    return class_weight # The output is a dictionary mapping letter grade to the corresponding weight

In [ ]:
params_dict = {'AA&S': [15, 120, 10],
               'AAS': [14, 120, 10],
               'CERT': [15, 160, 11],
               'CSC': [16, 100, 17]}

for r in ['AA&S', 'AAS', 'CERT', 'CSC']:

    # Load the training/validation sample
    print(r)
    train_df = train_df_all[train_df_all.deglvl == r]
    test_df = test_df_all[test_df_all.deglvl == r]
    print(train_df.shape,test_df.shape)

    train_df.loc[:,['vccsid']].to_stata("train_id_{}.dta".format(r), write_index=False)
    test_df.loc[:,['vccsid']].to_stata("test_id_{}.dta".format(r), write_index=False)
    
    optimal_d = params_dict[r][0]
    optimal_n = params_dict[r][1]
    optimal_nf = params_dict[r][2]
    rf = RandomForestClassifier(n_estimators=optimal_n, criterion="entropy",
                                max_depth=optimal_d,
                                random_state=0, n_jobs=-1, max_features=optimal_nf,
                                class_weight = calc_cw(train_df.grad_6years))
    rf.fit(train_df.loc[:,predictors], train_df.grad_6years)
    
    # Coefficients and predicted scores
    y_test_pred_rf = rf.predict_proba(test_df.loc[:,predictors])[:,1]
    y_train_pred_rf = rf.predict_proba(train_df.loc[:,predictors])[:,1]
    pickle.dump(y_test_pred_rf, open("y_test_pred_rf_{}.p".format(r), "wb"))
    pickle.dump(list(test_df.grad_6years), open( "y_test_{}.p".format(r), "wb"))
    pickle.dump(y_train_pred_rf, open("y_train_pred_rf_{}.p".format(r), "wb"))
    pickle.dump(list(train_df.grad_6years), open("y_train_{}.p".format(r), "wb"))
    print("Random Forest:")
    print("Validation AUC = {}".format(round(roc_auc_score(test_df.grad_6years, y_test_pred_rf),4)))
    print("Training AUC = {}".format(round(roc_auc_score(train_df.grad_6years, y_train_pred_rf),4)))
    
    race_column = []
    for i in range(test_df.shape[0]):
        if test_df.white.iloc[i] == 1:
            race_column.append("white")
        elif test_df.afam.iloc[i] == 1:
            race_column.append("afam")
        elif test_df.hisp.iloc[i] == 1:
            race_column.append("hisp")
        elif test_df.asian.iloc[i] == 1:
            race_column.append("asian")
        elif test_df.other.iloc[i] == 1:
            race_column.append("other")
        else:
            race_column.append("mi")
    race_column = np.array(race_column)
    pred_y = np.array(y_test_pred_rf)
    test_y = np.array(test_df.grad_6years)
    pred_y = pred_y[race_column != "mi"]
    test_y = test_y[race_column != "mi"]
    race_column = race_column[race_column != "mi"]
    print(len(race_column), len(pred_y), len(test_y))

    pred_score_by_race = pd.DataFrame({'race_column': race_column, 'pred_y': pred_y, 'test_y': test_y})
    pred_score_by_race.to_csv("pred_score_by_race_{}.csv".format(r), index=False)